In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import h5py

In [24]:

# fname1 = 'SPU_test2026-07-07_15-06-30.h5'
fname1 = '/home/gabriel.ascencao/2026-06-15/SPU_OS_eScan_10.0keV_h5_2026-06-15_22-09-29.h5'
fname2 = '/home/gabriel.ascencao/2026-06-15/SPU_CS_eScan_10.0keV_h5_2026-06-15_22-17-32.h5'


In [25]:

def read_data(fname):
    
    energy = []
    flux = []
    exp_time = []
    images = {} 
    max_intensity = []
    max_intensity_global = 0

    with h5py.File(fname, 'r') as f:
        group = f['images']
        keys = list(group.keys())

        for i in range(len(keys)):
            energy.append(group[keys[i]].attrs['energy_mon'])
            flux.append(group[keys[i]].attrs['flux_mon'])
            exp_time.append([group[keys[i]].attrs['exp_time_mon'],
                            group[keys[i]].attrs['exp_time_sp']])
            images[keys[i]] = np.array(group[keys[i]])
            max_intensity.append(group[keys[i]].attrs['max_intensity'])
            max_intensity_global = np.max([max_intensity_global, 
                                    group[keys[i]].attrs['max_intensity']])

    print('global max. intensity = {:0d}/1024 counts'.format(max_intensity_global))

    energy = np.array(energy)
    exp_time = np.array(exp_time)
    flux = np.array(flux) / exp_time[:,0]
    # flux /= np.max(flux)
    max_intensity = np.array(max_intensity)
    
    return energy, flux, images


In [26]:
energy1, flux1, images1 = read_data(fname1)
energy2, flux2, images2 = read_data(fname2)

global max. intensity = 991/1024 counts
global max. intensity = 960/1024 counts


In [27]:
energy1

array([ 8.99998569,  9.1749506 ,  9.34989452,  9.52503872,  9.69996166,
        9.70001507,  9.71205425,  9.7239275 ,  9.73585701,  9.74797058,
        9.75987053,  9.77199936,  9.7838583 ,  9.79586601,  9.80794048,
        9.82001781,  9.83199692,  9.84386826,  9.85594368,  9.86787415,
        9.87995434,  9.89194298,  9.9038229 ,  9.91602135,  9.92780113,
        9.94008827,  9.95192623,  9.96383095,  9.97598267,  9.98787022,
       10.00009918, 10.0119791 , 10.02381134, 10.03604603, 10.04781151,
       10.06001949, 10.07198715, 10.0839262 , 10.09601974, 10.10791874,
       10.1199255 , 10.13187408, 10.14397717, 10.1558857 , 10.16808701,
       10.18000317, 10.19181919, 10.20387173, 10.21583462, 10.2280941 ,
       10.24000454, 10.25187397, 10.26400089, 10.27607822, 10.28799248,
       10.30009556, 10.39997768, 10.51999283, 10.64009094, 10.76005745,
       10.88004398, 11.00001431, 11.11993504, 11.23991489, 11.35995483,
       11.47997379, 11.59981918, 11.70009136, 11.71481991, 11.73

Animated Plot

In [22]:
%matplotlib qt5

energy = energy1
flux = flux1
images = images1

# Setup figure with two equal-sized panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4), 
                               gridspec_kw={'width_ratios': [1, 1]})

line, = ax1.plot([], [], '.-k')
ax1.set_xlim(energy.min(), energy.max())
ax1.set_ylim(0, np.max(flux))
ax1.set_title("Flux vs Energy")
ax1.set_xlabel('Energy [keV]')
ax1.set_ylabel('Flux [a.u.]')

# Initialize the image display with the first image
first_img = images['img_001']
im = ax2.imshow(first_img, cmap='viridis', origin='lower', vmin=0, vmax=999)
ax2.set_title("Image")
ax2.set_xlabel('X [px]')
ax2.set_ylabel('Y [px]')

# Add colorbar with same height as ax2
cbar = fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
cbar.set_label("Intensity [a.u.]")

# Initialization function
def init():
    line.set_data([], [])
    im.set_data(np.zeros_like(first_img))
    return line, im

# Update function for animation
def update(frame):
    line.set_data(energy[:frame], flux[:frame])
    key = 'img_{0:03d}'.format(frame + 1)
    if key in images:
        im.set_data(images[key])
    return line, im

# Create animation
ani = FuncAnimation(fig, update, frames=len(energy) + 1,
                    init_func=init, blit=False, interval=300)

fig.tight_layout()

# ani.save("spectrum_measurement.gif", writer="ffmpeg", fps=5, dpi=150)




PLOT FLUX ONLY

In [14]:

fig, ax = plt.subplots()
ax.plot(energy, flux, 'o-')
ax.set_xlabel('Energy [keV]')
ax.set_ylabel('Flux [a.u.]')
ax.grid(alpha=0.3)
ax.minorticks_on()
ax.tick_params(which='both', axis='both', direction='in', top=True, right=True)
plt.show()

PLOT BOTH CURVES

In [31]:
fig, ax = plt.subplots(figsize=(4.5, 3.0))
ax.plot(energy1, flux1, '-', label='Open Slits')
ax.plot(energy2, flux2, '-', label='Closed Slits')
ax.set_xlabel('Energy [keV]')
ax.set_ylabel('Flux [a.u.]')
ax.grid(alpha=0.3)
ax.legend()
ax.minorticks_on()
ax.tick_params(which='both', axis='both', direction='in', top=True, right=True)
fig.tight_layout()
# plt.savefig('spectrum_slit_comparison.png', dpi=300)
plt.show()

In [29]:

flux1_aux = flux1-np.min(flux1)
flux2_aux = flux2-np.min(flux2)
fig, ax = plt.subplots(figsize=(4.5, 3.0))
ax.plot(energy1, flux1_aux/np.max(flux1_aux), '-', label='Open Slits')
ax.plot(energy2, flux2_aux/np.max(flux2_aux), '-', label='Closed Slits')
ax.set_xlabel('Energy [keV]')
ax.set_ylabel('Flux [a.u.]')
ax.grid(alpha=0.3)
ax.legend()
# ax.minorticks_on()
ax.tick_params(which='both', axis='both', direction='in', top=True, right=True)
fig.tight_layout()
# plt.savefig('spectrum_slit_comparison.png', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.imshow(images['img_035'])

In [9]:
plt.figure()
plt.plot(energy, exp_time[:,0])

In [22]:
plt.figure()
plt.plot(energy, max_intensity)